# 07 — Memória com ConversationBufferMemory

Neste notebook vamos estudar o conceito de **memória em aplicações conversacionais** usando `ConversationBufferMemory`.

Até agora, nossas chamadas para a LLM eram independentes:

```text
Pergunta 1 → LLM → Resposta 1
Pergunta 2 → LLM → Resposta 2
```

A LLM não recebe automaticamente a pergunta anterior.

Com memória:

```text
Mensagem 1
   ↓
Memória
   ↓
Mensagem 2 + histórico
   ↓
LLM
```

Assim, a aplicação consegue manter o contexto da conversa.


## 1. O que a memória resolve?

Imagine:

```text
Usuário: Meu nome é Felipe.
IA: Prazer, Felipe!

Usuário: Qual é meu nome?
IA: Não sei.
```

Isso pode acontecer porque, na segunda chamada, a aplicação não enviou o histórico anterior para o modelo.

A memória resolve isso armazenando o histórico e disponibilizando-o para as próximas interações.


## 2. O que é ConversationBufferMemory?

`ConversationBufferMemory` é uma implementação de memória que mantém o histórico da conversa em um buffer.

Podemos imaginar:

```text
Memória
├── Usuário: Meu nome é Felipe.
├── IA: Prazer, Felipe!
├── Usuário: Estou estudando IA.
└── IA: Legal!
```

Nas próximas interações, esse histórico pode ser incluído no contexto enviado para a LLM.

> **Observação:** `ConversationBufferMemory` e `ConversationChain` fazem parte da API clássica/legada do LangChain. Como essa é a API apresentada na aula, vamos estudá-la para entender o conceito. Em aplicações novas, a memória/conversação é normalmente construída com abstrações mais modernas.


## 3. Importando os componentes

Como estamos utilizando as APIs clássicas, memória e `ConversationChain` vêm de `langchain_classic`.

`ChatOllama` continua vindo de `langchain_ollama`.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain


## 4. Configurando o Qwen

Vamos continuar usando o Qwen local através do Ollama.


In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)


## 5. Criando a memória

Agora criamos nossa `ConversationBufferMemory`.


In [ ]:
memory = ConversationBufferMemory(
    return_messages=True
)


### O que significa `return_messages=True`?

Esse parâmetro faz com que o histórico seja representado como mensagens estruturadas, em vez de apenas uma string concatenada.

Isso é especialmente útil quando estamos trabalhando com modelos de chat.


## 6. Criando a ConversationChain

Agora conectamos a LLM com a memória através de uma `ConversationChain`.

A ideia é que a chain cuide do fluxo da conversa e utilize a memória para manter o histórico.


In [ ]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)


O fluxo fica:

```text
Usuário
   ↓
ConversationChain
   ↕
Memory
   ↓
ChatOllama
   ↓
Qwen
```


## 7. Primeira mensagem

Vamos informar nosso nome e o assunto que estamos estudando.


In [ ]:
resposta = conversation.invoke(
    "Meu nome é Felipe e estou estudando inteligência artificial."
)

print(resposta["response"])


A informação foi enviada para a LLM e também passou a fazer parte do histórico da memória.


## 8. Segunda mensagem

Agora vamos fazer uma pergunta que depende da informação anterior.


In [ ]:
resposta = conversation.invoke(
    "O que eu estou estudando?"
)

print(resposta["response"])


O ponto importante é que não repetimos a informação anterior.

A memória permite que a chain utilize o histórico para construir o contexto enviado ao modelo.


## 9. Inspecionando a memória

Podemos consultar o conteúdo armazenado pela memória.


In [ ]:
print(memory.load_memory_variables({}))


O resultado permite observar as mensagens armazenadas durante a conversa.


## 10. Adicionando mais contexto

Vamos continuar a conversa:


In [ ]:
resposta = conversation.invoke(
    "Também estou aprendendo Python para trabalhar com IA."
)

print(resposta["response"])


In [ ]:
resposta = conversation.invoke(
    "Quais são as duas coisas que estou estudando?"
)

print(resposta["response"])


## 11. Limpando a memória

Se quisermos começar uma nova conversa, podemos limpar o histórico.


In [ ]:
memory.clear()


In [ ]:
print(memory.load_memory_variables({}))


Depois de limpar a memória, o histórico anterior não estará mais disponível para a `ConversationChain`.


## 12. Como a memória funciona conceitualmente?

É importante entender que a memória não faz o modelo realmente "lembrar" sozinho.

A aplicação mantém o histórico e o utiliza para fornecer contexto às próximas chamadas.

```text
Mensagem 1
    ↓
Memória armazena
    ↓
Mensagem 2
    ↓
Histórico + Mensagem 2
    ↓
LLM
    ↓
Resposta
    ↓
Memória atualizada
```

A memória da aplicação é diferente de o modelo ter seus pesos alterados ou "aprender" permanentemente uma informação.


## 13. Código completo


In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain


llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)

memory = ConversationBufferMemory(
    return_messages=True
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

resposta = conversation.invoke(
    "Meu nome é Felipe e estou estudando inteligência artificial."
)

print("Resposta 1:")
print(resposta["response"])

resposta = conversation.invoke(
    "O que eu estou estudando?"
)

print("\nResposta 2:")
print(resposta["response"])
